Importamos los paquetes necesarios

In [9]:
import time
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

Configuración de los parámetros

In [10]:
CHAR_PATHS = ["./assets/junimo_game/junimo.png", "./assets/junimo_game/Garden_Pot.png"]
HAND_MODEL = "./models/hand_landmarker.task"

CHAR_H = 80 # alto del personaje
CHAR_X = 0.25
HIT_SCALE = 1.0 # escala del hitbox respecto al sprite (1 = hitbox ajustada al sprite, <1 = hitbox más pequeña)
HIT_USE_ALPHA = True
SMOOTH = 0.6 # seguimiento del dedo (1 = pegado al dedo, más bajo = más suave)

# Parámetros de obstaculos y velocidad del juego
PIPE_W = 100
FLIP_TOP = True
MARGIN_MIN = 40
OBELISK_PATHS = ["./assets/junimo_game/Water_Obelisk.png", "./assets/junimo_game/Desert_Obelisk.png",
                 "./assets/junimo_game/Earth_Obelisk.png", "./assets/junimo_game/Island_Obelisk.png"]
GAP0, GAP_MIN = 190, 140
SPACING = 280
SPEED0, SPEED_UP, SPEED_MAX = 160, 8, 380

Se implementan funciones para la lógica de juego.

In [11]:
rng = np.random.default_rng()


def load_character(path):
    im = cv2.imread(path, cv2.IMREAD_UNCHANGED)

    if im.shape[2] == 3:
        a = (~np.all(im > 240, axis=2)).astype(np.uint8) * 255
        im = np.dstack([im, a])
    return im


def prepare(im):
    f = im.astype(np.float32)
    f[:, :, :3] *= f[:, :, 3:4] / 255.0
    return f


def overlay(frame, sprite, src_pt, dst_pt, s, angle_deg):
    """Pinta el sprite (float32 premultiplicado): src_pt cae en dst_pt, con escala s y rotación."""
    a = np.radians(angle_deg)
    A = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]]) * s
    t = np.asarray(dst_pt, float) - A @ np.asarray(src_pt, float)
    h, w = sprite.shape[:2]
    corners = np.array([[0, 0], [w, 0], [w, h], [0, h]], float) @ A.T + t
    x0, y0 = np.floor(corners.min(0)).astype(int)
    x1, y1 = np.ceil(corners.max(0)).astype(int)
    H, W = frame.shape[:2]
    x0, y0, x1, y1 = max(x0, 0), max(y0, 0), min(x1, W), min(y1, H)
    if x1 <= x0 or y1 <= y0:
        return
    M = np.hstack([A, (t - [x0, y0])[:, None]])
    warped = cv2.warpAffine(sprite, M, (x1 - x0, y1 - y0), flags=cv2.INTER_LINEAR,
                            borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0, 0))
    roi = frame[y0:y1, x0:x1]
    roi[:] = np.clip(warped[:, :, :3] + roi * (1 - warped[:, :, 3:4] / 255.0), 0, 255).astype(np.uint8)


# Lógica de juego

def blit(frame, sprite, x, y):
    """Pega un sprite (float32 premultiplicado) con su esquina superior izquierda en (x, y), recortando lo que se sale."""
    h, w = sprite.shape[:2]
    H, W = frame.shape[:2]
    x0, y0, x1, y1 = max(x, 0), max(y, 0), min(x + w, W), min(y + h, H)
    if x1 <= x0 or y1 <= y0:
        return
    s = sprite[y0 - y:y1 - y, x0 - x:x1 - x]
    roi = frame[y0:y1, x0:x1]
    roi[:] = np.clip(s[:, :, :3] + roi * (1 - s[:, :, 3:4] / 255.0), 0, 255).astype(np.uint8)
    

def new_game(W, H):
    return dict(W=W, H=H, x=CHAR_X * W, y=H / 2, vy=0.0, pipes=[], score=0, over=False, started=False)


def make_hitbox(im):
    """Hitbox rectangular del personaje según su imagen y la escala con la que se dibuja (CHAR_H)."""
    h, w = im.shape[:2]
    s = CHAR_H / h
    if HIT_USE_ALPHA:
        ys, xs = np.where(im[:, :, 3] > 20)
        x0, x1, y0, y1 = xs.min(), xs.max() + 1, ys.min(), ys.max() + 1
    else:
        x0, x1, y0, y1 = 0, w, 0, h
    return dict(w=(x1 - x0) * s * HIT_SCALE, h=(y1 - y0) * s * HIT_SCALE,
                dx=((x0 + x1) / 2 - w / 2) * s, dy=((y0 + y1) / 2 - h / 2) * s)


def rects_overlap(ax0, ay0, ax1, ay1, bx0, by0, bx1, by1):
    return ax0 < bx1 and ax1 > bx0 and ay0 < by1 and ay1 > by0


def update(g, finger, dt):
    """finger = (x, y) de la punta del índice en px, o None si no hay mano (el juego se pausa)."""
    if g["over"] or finger is None:
        return
    W, H = g["W"], g["H"]
    g["started"] = True

    prev = g["y"]
    g["y"] += SMOOTH * (finger[1] - g["y"])
    g["y"] = min(max(g["y"], CHAR_H / 2), H - CHAR_H / 2)
    g["vy"] = (g["y"] - prev) / max(dt, 1e-3)

    # Generación de obstáculos
    if not g["pipes"] or g["pipes"][-1]["x"] < W - SPACING:
        gap = max(GAP_MIN, GAP0 - 4 * g["score"])
        margin = max(MARGIN_MIN, H - gap - OBS_H)      # así el obelisco siempre llega hasta el borde de la pantalla
        g["pipes"].append(dict(x=float(W), gap=gap, gap_y=rng.uniform(gap / 2 + margin, H - gap / 2 - margin), passed=False,
                               img_t=int(rng.integers(len(OBELISKS))), img_b=int(rng.integers(len(OBELISKS)))))
    speed = min(SPEED0 + SPEED_UP * g["score"], SPEED_MAX)
    for p in g["pipes"]:
        p["x"] -= speed * dt
        if not p["passed"] and p["x"] + PIPE_W < g["x"]:
            p["passed"] = True
            g["score"] += 1
    g["pipes"] = [p for p in g["pipes"] if p["x"] + PIPE_W > 0]

    # Cálculo de colisiones
    cx, cy = g["x"] + HIT["dx"], g["y"] + HIT["dy"]
    bx0, by0, bx1, by1 = cx - HIT["w"] / 2, cy - HIT["h"] / 2, cx + HIT["w"] / 2, cy + HIT["h"] / 2
    for p in g["pipes"]:
        top, bot = p["gap_y"] - p["gap"] / 2, p["gap_y"] + p["gap"] / 2
        if (rects_overlap(bx0, by0, bx1, by1, p["x"], 0, p["x"] + PIPE_W, top) or
                rects_overlap(bx0, by0, bx1, by1, p["x"], bot, p["x"] + PIPE_W, H)):
            g["over"] = True


# Función auxiliar de dibujado del juego 

def draw_game(frame, g, sprite, finger, best, show_hitbox=False):
    H = g["H"]
    for p in g["pipes"]:
        x0 = int(round(p["x"]))
        top, bot = int(p["gap_y"] - p["gap"] / 2), int(p["gap_y"] + p["gap"] / 2)
        blit(frame, OBELISKS[p["img_b"]][0], x0, bot)
        spr_t = OBELISKS[p["img_t"]][1 if FLIP_TOP else 0]
        blit(frame, spr_t, x0, top - spr_t.shape[0])

    h = sprite.shape[0]
    angle = float(np.clip(g["vy"] * 0.04, -25, 25))
    overlay(frame, sprite, (sprite.shape[1] / 2, h / 2), (g["x"], g["y"]), CHAR_H / h, angle)
    if show_hitbox:
        cx, cy = g["x"] + HIT["dx"], g["y"] + HIT["dy"]
        cv2.rectangle(frame, (int(cx - HIT["w"] / 2), int(cy - HIT["h"] / 2)), (int(cx + HIT["w"] / 2), int(cy + HIT["h"] / 2)), (0, 0, 255), 2)

    cv2.putText(frame, f"Puntos: {g['score']}   Mejor: {best}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    
    if finger is None and not g["over"]:
        cv2.putText(frame, "PAUSA: no se ha detectado mano/nariz", (10, g["H"] - 20), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    if g["over"]:
        frame[:] = (frame * 0.4).astype(np.uint8)
        cv2.putText(frame, "GAME OVER", (g["W"] // 2 - 150, g["H"] // 2 - 10), cv2.FONT_HERSHEY_SIMPLEX, 1.6, (0, 0, 255), 4)
        cv2.putText(frame, f"{g['score']} puntos - R para reiniciar", (g["W"] // 2 - 190, g["H"] // 2 + 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

Asimismo, se realizará la carga del "jugador" y de los obstáculos

In [12]:
CHARS = []
for path in CHAR_PATHS:
    im = load_character(path)
    CHARS.append((prepare(im), make_hitbox(im)))

char_i = 0
sprite, HIT = CHARS[char_i]

OBELISKS = []
for path in OBELISK_PATHS:
    im = load_character(path)
    s = PIPE_W / im.shape[1]
    im = cv2.resize(im, (PIPE_W, int(round(im.shape[0] * s))), interpolation=cv2.INTER_AREA)
    OBELISKS.append((prepare(im), prepare(cv2.flip(im, 0))))
OBS_H = min(o[0].shape[0] for o in OBELISKS) 

Bucle de ejecución

In [13]:
hands = vision.HandLandmarker.create_from_options(vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=HAND_MODEL),
    running_mode=vision.RunningMode.VIDEO, num_hands=1))

vid = cv2.VideoCapture(0)
vid.set(3, 640)
vid.set(4, 480)

g, best, show_hitbox = None, 0, True
last = time.perf_counter()

while True:
    ret, frame = vid.read()

    if not ret:
        break

    frame = cv2.flip(frame, 1)
    H, W = frame.shape[:2]
    if g is None:
        g = new_game(W, H)

    now = time.perf_counter()
    dt, last = min(now - last, 0.1), now

    # Se toma como referencia la punta del dedo índice (landmark 8)
    res = hands.detect_for_video(
        mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)), int(now * 1000))
    finger = None
    
    if res.hand_landmarks:
        lm = res.hand_landmarks[0][8]
        finger = (lm.x * W, lm.y * H)

    update(g, finger, dt)
    best = max(best, g["score"])
    draw_game(frame, g, sprite, finger, best, show_hitbox)
    if finger is not None:
        cv2.circle(frame, (int(finger[0]), int(finger[1])), 6, (0, 255, 255), -1)   # Punta del índice
    cv2.imshow("Junimo Fly - Mano", frame)

    tecla = cv2.waitKey(1) & 0xFF
    if tecla in (ord("q"), 27):
        break
    elif tecla == ord("r"):
        g = new_game(W, H)
    elif tecla == ord("d"):
        show_hitbox = not show_hitbox
    elif tecla == ord("c") and (g["over"] or not g["started"]):
        char_i = (char_i + 1) % len(CHARS)
        sprite, HIT = CHARS[char_i]

hands.close()
vid.release()
cv2.destroyAllWindows()

Variante 2: Junimo Fly con nariz

In [14]:
FACE_MODEL = "./models/face_landmarker.task"
NOSE_GAIN = 2.5 # Cuánto se amplifica el movimiento de la nariz
SMOOTH = 0.4     

In [15]:
def nose_control(nose, origin, W, H):
    """Desplazamiento de la nariz respecto al punto neutro, amplificado -> punto de control en pantalla."""
    d = (nose - origin) * NOSE_GAIN
    return (CHAR_X * W + d[0], H / 2 + d[1])

In [16]:
face = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=FACE_MODEL),
    running_mode=vision.RunningMode.VIDEO, num_faces=1))

vid = cv2.VideoCapture(0)
vid.set(3, 640)
vid.set(4, 480)

g, best, show_hitbox = None, 0, True
origin, nose_s = None, None
last = time.perf_counter()

while True:
    ret, frame = vid.read()

    if not ret:
        break

    frame = cv2.flip(frame, 1)
    H, W = frame.shape[:2]

    if g is None:
        g = new_game(W, H)

    now = time.perf_counter()
    dt, last = min(now - last, 0.1), now

    # Se toma como referencia la punta de la nariz (landmark 1 de la malla facial)
    res = face.detect_for_video(
        mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)), int(now * 1000))
    face_control = None
    
    if res.face_landmarks:
        n = res.face_landmarks[0][1]
        nose = np.array([n.x * W, n.y * H])
        nose_s = nose if nose_s is None else nose_s + 0.5 * (nose - nose_s) # Suavizado ligero
        if origin is None:
            origin = nose_s.copy() # El origen es la primera posición de la nariz detectada
        face_control = nose_control(nose_s, origin, W, H)

    update(g, face_control, dt)
    best = max(best, g["score"])
    draw_game(frame, g, sprite, face_control, best, show_hitbox)
    if face_control is not None:
        cv2.circle(frame, (int(nose_s[0]), int(nose_s[1])), 6, (0, 255, 255), -1)  

    cv2.imshow("Junimo Fly - Cara", frame)

    tecla = cv2.waitKey(20) & 0xFF
    if tecla == 27:
        break
    elif tecla == ord("r"):
        g = new_game(W, H)
        origin = None
    elif tecla == ord("d"):
        show_hitbox = not show_hitbox
    elif tecla == ord("c") and (g["over"] or not g["started"]):
        char_i = (char_i + 1) % len(CHARS)
        sprite, HIT = CHARS[char_i]

face.close()
vid.release()
cv2.destroyAllWindows()